# Ngày 3 — Conversational AI (AI hội thoại) — tức Chatbot!

In [1]:
# imports (nhập thư viện)

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Nạp environment variables (biến môi trường) từ file .env
# In prefix (tiền tố) của keys để hỗ trợ debug

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("OpenAI API Key chưa được đặt")

OpenAI API Key tồn tại và bắt đầu bằng sk-proj-


In [3]:
# Khởi tạo

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [4]:
# Lại ở scientist-mode (chế độ nhà khoa học) — tôi sẽ đổi biến global này trong lab

system_message = "Bạn là một trợ lý hữu ích"

## Bây giờ, viết một callback (hàm gọi lại) mới

Chúng ta cần viết một function tên:

`chat(message, history)`

Đây sẽ là callback mà chúng ta đưa cho Gradio.

### Nhiệm vụ của function này

Nhận một message (tin nhắn), nhận cuộc hội thoại trước đó, rồi trả về response (phản hồi).


In [5]:
def chat(message, history):
    return "chuối"

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [7]:
def chat(message, history):
    return f"Bạn vừa nói {message} và history là {history} nhưng tôi vẫn nói chuối"

In [8]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## Được rồi! Viết một chat callback tốt hơn một chút!

In [9]:
# Callback chat thật: ghép system + history + message rồi gọi OpenAI

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [11]:
# Cùng callback, nhưng stream (luồng) từng chunk về Gradio

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


## Tiếp tục!

Dùng system message để thêm context (ngữ cảnh), và đưa một example (ví dụ) câu trả lời — đây lại là "one shot prompting" (prompt một ví dụ).

In [13]:
system_message = "Bạn là trợ lý hữu ích tại một cửa hàng quần áo. Bạn nên nhẹ nhàng khuyến khích \
khách hàng thử các món đang sale (giảm giá). Mũ giảm 60%, hầu hết món khác giảm 50%. \
Ví dụ, nếu khách nói 'Tôi muốn mua một chiếc mũ', \
bạn có thể trả lời kiểu: 'Tuyệt quá — chúng tôi có rất nhiều mũ, kể cả vài mẫu trong sự kiện sale.'\
Nếu khách chưa biết mua gì, hãy khuyến khích họ mua mũ."

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [15]:
system_message += "\nNếu khách hỏi giày, hãy trả lời rằng hôm nay giày không sale, \
nhưng nhắc khách xem mũ!"

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [17]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower() or 'thắt lưng' in message.lower():
        relevant_system_message += " Cửa hàng không bán thắt lưng; nếu bị hỏi về thắt lưng, hãy giới thiệu các món khác đang sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [18]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh</h2>
            <span style="color:#181;">Conversational Assistants (trợ lý hội thoại) tất nhiên là use case (tình huống dùng) rất phổ biến của Gen AI, và các frontier models mới nhất rất giỏi hội thoại có sắc thái. Gradio giúp dễ có user interface (giao diện người dùng). Một kỹ năng then chốt khác chúng ta đã học là dùng prompting để cung cấp context, thông tin và examples.
<br/><br/>
Hãy nghĩ cách áp dụng AI Assistant cho công việc của bạn, rồi tự làm một prototype (bản mẫu). Dùng system prompt để đưa context về doanh nghiệp, và đặt giọng điệu cho LLM.</span>
        </td>
    </tr>
</table>